**This notebook is a test environment for the ruleset-engine program. It can be used to test overall program architecture, logical flow and execution strategy of ruleset-engine. Note: ruleset-engine is a subset of the ruleset framework.**

In [1]:
import sys
sys.path.insert(0, "/home/anjan_purkayastha/Documents/openboxbio/20211227_dsl-for-data-validation/code/ruleset_engine")

In [2]:
from textx import get_location, TextXSyntaxError
from textx.metamodel import metamodel_from_file
import pandas as pd
import numpy as np
from pandas._libs.parsers import STR_NA_VALUES
from ruleset import set_value_rules as s
from ruleset import column_rules as c
from ruleset import in_column_value_rules as i
from decimal import *
import re


In [13]:
rules_infile = "/home/anjan_purkayastha/Documents/openboxbio/20211227_dsl-for-data-validation/code/ruleset_engine/tests/test_conditional_rules_multiple_consequents_ruleset"
data_infile = "/home/anjan_purkayastha/Documents/openboxbio/20211227_dsl-for-data-validation/code/ruleset_engine/tests/test_ruleblocks.csv"

In [14]:
'''
Read the Ruleset metamodel from metamodel file.
Read the rules file.
'''
metamodel_infile = "./ruleset/RuleSet.tx"
mm = metamodel_from_file(metamodel_infile, autokwd=True)
try:
    data_rules = mm.model_from_file(rules_infile)
    print(f'{rules_infile} is ok.')
except TextXSyntaxError as err:
    print(f'Syntax Error found in {err.filename} in line {err.line}, column {err.col}')
    print(f'Error: {err.message}')

/home/anjan_purkayastha/Documents/openboxbio/20211227_dsl-for-data-validation/code/ruleset_engine/tests/test_conditional_rules_multiple_consequents_ruleset is ok.


In [15]:
'''
Create a dictionary of column rules and of column value rules from the parsed rules file.
Key = rule name; Value = rule object. 
'''
set_value_rules_dict = {}
column_rules_dict = {}
column_value_rules_dict = {}
conditional_rules_list = []
block_rules_list = []

set_value_rules_dict = {type(rule).__name__: rule for rule in data_rules.set_value_rules}
column_rules_dict = {type(rule).__name__: rule for rule in data_rules.column_rules}
column_value_rules_dict = {rule.column_name.name: rule for rule in data_rules.column_value_rules}
conditional_rules_list = data_rules.conditional_rules
block_rules_list = data_rules.block_rules

In [17]:
vars(conditional_rules_list[0])

{'rule_tag': <ConditionalRuleTag:HiRes Rule>,
 'column_value_rule_input_1': <textx:RuleSet.InColumnValueRule instance at 0x74aa8812eb10>,
 'column_value_rule_input_2': [<textx:RuleSet.InColumnValueRule instance at 0x74aa7cdebb30>,
  <textx:RuleSet.InColumnValueRule instance at 0x74aa7cdea000>],
 '_tx_position': 197,
 '_tx_position_end': 350,
 'parent': <textx:RuleSet.RuleSet instance at 0x74aa7d0da0c0>}

In [6]:
then_rule = vars(conditional_rules_list[1])['column_value_rule_input_2']

In [7]:
if_rule = vars(conditional_rules_list[1])['column_value_rule_input_1']

In [8]:
vars(if_rule)

{'column_name': <ColumnName:value1>,
 'setlocalrules': [],
 'valuerules': [<textx:RuleSet.HasValueRule instance at 0x74aa7d128a70>],
 '_tx_position': 253,
 '_tx_position_end': 281,
 'parent': <textx:RuleSet.ConditionalRule instance at 0x74aa7d0da750>}

In [22]:
column_value_rules_dict['value1'].setlocalrules[0].local_tolerance_input

IndexError: list index out of range

In [23]:
'''
Validate rules file. 
Checks:
(a) Check if list of column names exists. If not, then print error message and exit.
(b) Check if each column in the column names list has an associated set of value rules.  
(b) Check if each column with at least one value rule is listed in the columns list.
'''
#Extract column names into a list, if ColumnListRule exists. If not, print error message and exit.
try:
    column_names_list = [column.name for column in column_rules_dict['ColumnListRule'].column_names_list]
except KeyError as err:
    print("A list of column names is required in the rules file. Please enter a list of column names and try again." + "\n" + "Exiting RuleSet Data Validator.")
    sys.exit()
    
if column_value_rules_dict.keys():
    column_names_set = set(column_names_list) #create set of the list of column names
    columns_in_column_value_rules = column_value_rules_dict.keys()
    columns_in_column_value_rules_set = set(columns_in_column_value_rules) #create set of all column names with at least one value rule.
    
    columns_with_no_value_rules = column_names_set - columns_in_column_value_rules_set #create set with columns that have no value rules.
    columns_not_in_columns_list = columns_in_column_value_rules_set - column_names_set #create set of columns with value rule(s) but that are not included in list of column names.
    
    if (len(columns_with_no_value_rules) == 0):
        print("All columns in columns list have associated value rules.")
    else:
        for column_name in columns_with_no_value_rules:
            print(f'Warning: column {column_name} has no associated value rules.')
    
    if (len(columns_not_in_columns_list) == 0):
        print("All columns with at least one value rule are included in the columns list.")
    else:
        for column_name in columns_not_in_columns_list:
            print(f'Warning: column {column_name} has value rules, but is not in columns list.')
else:
    print('Warning: No column value rules included in rules file.')


All columns in columns list have associated value rules.
All columns with at least one value rule are included in the columns list.


In [24]:
'''
If null values list has been specified then a NullValuesRule object will be included in the rules file.
Use the set_custom_null_values to extract the null values list and add it to the default na values 
list to create the all_na_values list. Delete rule as it does not have to be run again.
If no NullValuesRule has been specified then the default na values list is assigned to all_na_values.
'''
try:
    input_rule = set_value_rules_dict['SetNullValuesRule']
    input_rule_name = type(input_rule).__name__
    all_na_values = s.set_values_dict[input_rule_name](input_rule)
except:
    all_na_values = STR_NA_VALUES

In [25]:
'''
READ INPUT FILE with data to be validated. 
all_na_values includes any na values specified in rules file.
Create a pandas dataframe from parsed data input file.
'''

if data_infile.lower().endswith(".xlsx"):
    data_df = pd.read_excel(
        data_infile,
        header=0,
        dtype=str,
        true_values = ['True', 'true', 'TRUE'],
        false_values = ['False', 'false', 'FALSE'],
        keep_default_na=False,
        na_values=all_na_values,
        engine='openpyxl'
    )
elif data_infile.lower().endswith(".csv"):
    data_df = pd.read_csv(
        data_infile,
        sep = ",",
        header=0,
        dtype=str,
        keep_default_na=False,
        na_values=all_na_values,
        skip_blank_lines=False
    )
else:
    raise ValueError(f"Unsupported file type: {data_infile}")

data_column_names = data_df.columns.to_list()

In [26]:
data_df

,value1,value2,value3,value4,value5,value6,value7,value8,value9,value10,value11,value12,value13,value14,value15,value16,value17,value18,value19,value20
0,ABC,ABC123,123XYZ,cat,1,abc,ABC123,a@test.com,12345,AB,color,A,A,cat,1A2B,ABC,E,2025-01-31,ABc1,USD100
1,ABC,ABC,XYZ,mycat,12345,AbcD,a1,john.doe@phfscience.nz,45678,ABCD,colour,AAAAA,A,dog,FFFF,123,ABCED,1917-12-12,TEST9,USD300
2,ab,XABC,XYZ123,CAT,12A,abc1,Abc-1,abc.com,123,ABCDE,colouur,AAAB,AAB,catdog,1G2B,A B,EE,2025/10/31,abc1,EUR100


In [27]:
'''
Print a summary of data in input file. 
TODO: Convert file summary into multiple lines in the run log file.
'''
#file_summary = c.h.summarize_infile(data_df, data_infile)
#file_summary['Columns']

'\nPrint a summary of data in input file. \nTODO: Convert file summary into multiple lines in the run log file.\n'

In [28]:
'''
For each column rule in the rules file, run the corresponding rule checker function.
'''
for column_rule in column_rules_dict.keys():
    rule = column_rules_dict[column_rule]
    if c.column_rule_checker_dict[column_rule]:
        ruleset_message = c.column_rule_checker_dict[column_rule](df = data_df, input_rule = rule, input_column_names_list = column_names_list, input_set_value_rules = set_value_rules_dict, input_column_rules = column_rules_dict, input_column_value_rules = column_value_rules_dict)

In [29]:
'''
Check the list of columns in data file against the columns with rules in the rules file.
Raise Warning if (a) column in data file has no rules or (b) a column with rules is not present in the data file.
'''
columns_with_rules_set = set(list(column_value_rules_dict.keys()))
data_column_names_set = set(data_column_names)
data_columns_with_rules = list(data_column_names_set & columns_with_rules_set)
columns_in_rules_but_not_in_data = list(columns_with_rules_set - data_column_names_set)
columns_in_data_but_not_in_rules = list(data_column_names_set - columns_with_rules_set)
columns_in_rules_but_not_in_data_warning_message = [f'Column {x} in rules file but not in data file.' for x in columns_in_rules_but_not_in_data]
columns_in_data_but_not_in_rules_warning_message = [f'Column {x} in data file but not in rules file.' for x in columns_in_data_but_not_in_rules]

In [30]:
'''
For each column value rule in the rules file, that is present in the data file, run the corresponding column value rule checker function.
'''
for column_name in data_columns_with_rules:
        value_rules_list = column_value_rules_dict[column_name].valuerules
        set_local_rules_list = column_value_rules_dict[column_name].setlocalrules
        local_rules_dict = {type(local_rule).__name__: local_rule for local_rule in set_local_rules_list}
        for value_rule in value_rules_list:
            value_rule_name = type(value_rule).__name__
            if i.in_column_value_rule_checker_dict[value_rule_name]:
                ruleset_message = i.in_column_value_rule_checker_dict[value_rule_name](df = data_df, input_rule = value_rule, input_column_name = column_name, input_set_value_rules = set_value_rules_dict, input_column_rules = column_rules_dict, input_column_value_rules = column_value_rules_dict, local_rules = local_rules_dict)
                print(column_name)
                print(ruleset_message[0])
                print(ruleset_message[1])

value16
[True, True, False]
['Error: column value16, row 3:Value A B does not match /^\\S+$/.']
value15
[True, True, False]
['Error: column value15, row 3:Value 1G2B does not match /^[0-9A-F]+$/.']
value8
[True, True, False]
['Error: column value8, row 3:Value abc.com does not match /^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$/.']
value12
[True, True, False]
['Error: column value12, row 3:Value AAAB does not match /^A+$/.']
value2
[True, True, False]
['Error: column value2, row 3:Value XABC does not match /^ABC.*/.']
value20
[True, True, False]
['Error: column value20, row 3:Value EUR100 does not match /(^USD)\\d+/.']
value10
[True, True, False]
['Error: column value10, row 3:Value ABCDE does not match /^[A-Z]{2,4}$/.']
value6
[True, True, False]
['Error: column value6, row 3:Value abc1 does not match /^[A-Za-z]+$/.']
value9
[True, True, False]
['Error: column value9, row 3:Value 123 does not match /^\\d{5}$/.']
value1
[True, True, False]
['Error: column value1, row 3:Value ab do

In [30]:
set_local_rules_list

[]

In [7]:
def validate_rule(rule_name, df, input_column_name, input_rule, input_set_value_rules, input_column_rules, input_column_value_rules):
    ruleset_message = ""
    valuerules_list = input_rule.valuerules
    if len(valuerules_list) > 1:
        ruleset_message = f'Conditional rule {rule_name} is invalid. Only one atomic rule per column is allowed.'
        return ruleset_message
    else:
        value_rule = valuerules_list[0]
        value_rule_name = type(value_rule).__name__
        if i.in_column_value_rule_checker_dict[value_rule_name]:
            [ruleset_message_booleans, ruleset_message_errors] = i.in_column_value_rule_checker_dict[value_rule_name](df = data_df, input_rule = value_rule, input_column_name = input_column_name, input_set_value_rules = set_value_rules_dict, input_column_rules = column_rules_dict, input_column_value_rules = column_value_rules_dict)
            ruleset_boolean_dict = {index + 1: value for index, value in enumerate(ruleset_message_booleans)}
        return ruleset_boolean_dict

In [8]:
"""
Validates conditional rules in RuleSet where the antecedent is a single atomic rule
and the consequent is a list of atomic rules.

This function iterates over a dictionary of conditional rules, where each rule maps an
"if" condition (a single atomic rule) to one or more "then" conditions (a list of atomic rules).
For each rule:
- It first checks which rows in the DataFrame satisfy the "if" condition.
- If no rows match the "if" condition, it logs an informational message and skips the rule.
- Otherwise, it validates the "then" conditions for the same rows and ensures the
  condition holds: *if condition is True, then all "then" conditions must also be True
  for the same rows*.
- Errors are logged if any rows satisfy the "if" condition but fail any of the "then" conditions.

Returns:
    None. 
    The function prints validation information and errors to standard output.
"""

for conditional_rule in conditional_rules_list:
    if_rule = conditional_rule.column_value_rule_input_1
    then_rule_list = conditional_rule.column_value_rule_input_2
    if_column_name = if_rule.column_name.name
    if_ruleset_message = validate_rule(rule_name = rule_name, df = data_df, input_column_name = if_column_name , input_rule = if_rule, input_set_value_rules = set_value_rules_dict, input_column_rules = column_rules_dict, input_column_value_rules = column_value_rules_dict) 
    if_is_true = {k:v for k,v in if_ruleset_message.items() if v is True}
    if not if_is_true:
        LOGFILE.write(f"Info: No value pairs in {if_column_name} and {then_column_name} to validate.")
        continue
    for then_rule in then_rule_list:
        then_column_name = then_rule.column_name.name
        then_ruleset_message = validate_rule(rule_name = rule_name, df = data_df, input_column_name = then_column_name , input_rule = then_rule, input_set_value_rules = set_value_rules_dict, input_column_rules = column_rules_dict, input_column_value_rules = column_value_rules_dict) 
        then_is_true = {k:v for k,v in then_ruleset_message.items() if v is True}
        if_then_match = set(if_is_true.keys()).issubset(then_is_true.keys())
        if if_then_match:
            LOGFILE.write(f"Info: conditional rule for columns: {if_column_name}, {then_column_name} is OK.")
            continue
        else:
            missing_keys = [key for key in if_is_true.keys() if key not in then_is_true.keys()]
            for key in missing_keys:
                LOGFILE.write(f"Error: conditional rule fails for columns: {if_column_name}, {then_column_name}, row {key}.")
            continue
        
    

NameError: name 'rule_name' is not defined